<a href="https://colab.research.google.com/github/PriFranca/Estudo-Ds/blob/master/nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redes Neurais Artificiais

## Preparando os dados

Vamos criar uma rede neural simples para prever *churn* de clientes.

Vamos come?ar pela leitura dos dados:

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/WoMakersCode/data-science-bootcamp/master/5.1%20Deep%20Learning/data/Churn_Modelling.csv')
df.head()

In [ ]:
df['Exited'].value_counts()

In [ ]:
df['Geography'].value_counts()

In [ ]:
df['Gender'].value_counts()

Para este exemplo vamos fazer um tratamento simples dos dados, apenas convertendo as vari?veis categoricas em dummies:

In [ ]:
df = pd.get_dummies(df, columns=['Geography', 'Gender'])
df.columns

Vamos separar os dados de teste e treinamento:

In [ ]:
X = df[['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember','EstimatedSalary',
        'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Gender_Female']]

y = df['Exited']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X , y, test_size = 0.1)
X_train, X_val, y_train, y_val = train_test_split(X_train , y_train, test_size = 0.1)

print(X_train.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
X_val = sc.transform(X_val)

## Construindo o modelo

Agora com o dados prontos vamos montar a nossa rede neural. Vamos usar a library [Keras](https://keras.io) rodando em cima do [TensorFlow](https://tensorflow.org/)


1. Defini??o da arquitetura

2. Compila??o

3. Treinamento

4. Avalia??o

### 1. Defini??o da arquitetura:
Definir a arquitetura da rede

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam

In [ ]:
def build_model():
    model = Sequential()

    #rede neural = varias pequenas regressoes
    # primeira camada adiciona o shape do input
    # adiciona a funcao de ativacao
    # quantidade de units (neur?nios) - colocar primeira camada numero maior primeiro. Se colocar 10 no input deve colocar 6, por exemplo no segundo)
    #input = variaveis, no caso do exemplo ser?o 12. input = primeira camada
    # tamb?m ? poss?vel alterar a inicializacao, bias, entre outros -- https://keras.io/layers/core/

    model.add(Dense(units=10, input_dim=12, activation='relu'))
    model.add(Dense(10, activation='relu'))

    #Camada de saida com o resultado de 1 classe e a ativa??o sigmoid -- outras fun??es de ativa??o: https://keras.io/activations/
    model.add(Dense(1, activation='sigmoid'))
    return model

### Vamos entender melhor as fun??es de ativa??o:

Em cada neur?nio da rede h? uma fun??o de ativa??o, que decide se o neur?nio deve ser *ativado*, e transmitir informa??es para a pr?xima camada.

![](https://i1.wp.com/deeplearningbook.com.br/wp-content/uploads/2018/02/act.png?w=406)

A fun??o mais comum nas camadas intermedi?rias ? a relu:

![](https://cdn-images-1.medium.com/max/937/1*oePAhrm74RNnNEolprmTaQ.png)

Na camada de sa?da a rede precisa nos retornar a probabilidade do cliente fazer o cancelamento.

Por ser uma probabilidade (de 0 a 1), n?s usamos a fun??o sigmoid:

![as vezes a fun??o sigm?ide ? simplesmente representada pela curva S](https://sabedoriararefeita.files.wordpress.com/2016/02/ann_sigmoid.png?w=615)


Outras fun??es comuns:

Softmax -> Usada na camada de output para problemas de multiclasse, a soma das probabilidades de todas as classes dar? 1.

elu -> para ser usada nas camadas intermediarias no lugar da relu, uma exponencial ? aplicada nos valores menores que 0.

> Em regress?o n?o h? fun??o de ativa??o na camada de output

outras fun??es de ativa??o: https://keras.io/activations/
explica??es extras: http://deeplearningbook.com.br/funcao-de-ativacao/

In [ ]:
#multiclasse= varias classes e quero sabe somente um ( uma imagem com varios animais e quero saber qual deles ? o gato)
#multilabel =

In [ ]:
model = build_model()

In [ ]:
model.summary()

In [ ]:
#quantidade de pesos que ela tem que aprender (no caso a primeira camada esta 130 pesos. 12 inputs x 10 layers)

> Podemos ver que na primeira camada 130 par?metros (pesos) ser?o aprendidos ((12 inputs x 10 layers) + (1 bias * 10 layers))

### 2. Compilar o modelo:

Definer como a rede ir? aprender. Qual o otimizador com os par?metros de learning rate, fun??o e parametros espec?ficos da fun??o e a loss function.


In [ ]:
# outras fun??es de loss: https://keras.io/losses/
# outros optimizers: https://keras.io/optimizers/
adam = Adam(lr=0.01)
model.compile(loss='binary_crossentropy',
             optimizer=adam,
             metrics=['accuracy'])

### Vamos entender como a rede aprende:

Para aprender os par?metros $w$ e $b$ ? preciso uma **fun??o de custo**. Primeiro, vamos definir uma fun??o de perda ou $Loss Function$ de modo que quanto mais pr?ximo da resposta certa, menor seja o valor dessa fun??o:

$L(\hat{y},y)=-(y\log{\hat{y}} + (1-y)\log{(1-\hat{y})})$ (binary_crossentropy)

> Se uma inst?ncia tem label 1, ent?o $(1-y)$ ? $0$, deixando apenas o lado esquerdo da equa??o. Pra que ele seja o menor poss?vel, $\hat{y}$ precisa ser o maior poss?vel, no caso o mais pr?ximo de 1. O oposto tamb?m se aplica para quando o label ? 0.

Com isso, temos a func?o de custo:

$J(w,b)=\frac{1}{m}\sum_{i=1}^{m}L(\hat{y}^i,y^i)$

Dado nosso custo, queremos encontrar $w$ e $b$ que minimize esse custo. Para isso utilizamos o **Gradiente Descendente**. A fun??o de custo ? uma func?o convexa, como uma bacia, ent?o o que o gradiente faz ? ir descendo o mais r?pido poss?vel at? chegar no fundo da bacia, no menor ponto, independente do ponto inicial.

![enter image description here](https://blog.paperspace.com/content/images/2018/05/68747470733a2f2f707669676965722e6769746875622e696f2f6d656469612f696d672f70617274312f6772616469656e745f64657363656e742e676966.gif)

Para fazer essa "decida", utilizaremos a derivada do custo e uma taxa de aprendizado ou *learning rate*, da seguinte forma:

A cada itera??o do algoritmo temos $w = w - \alpha \frac{\mathrm{d}J}{\mathrm{d}w}$, sendo $\alpha$ o learning rate.

De modo geral, atualizamos w e b a cada itera??o, sendo a velocidade controlada pelo learning rate, at? chegarmos no ponto m?nimo de custo.

**Mas o que ? o Adam ent?o?**

Algoritmo de otimiza??o da taxa de aprendizado adapt?vel que foi projetado especificamente para o treinamento de redes neurais profundas, pode ser usado em vez do procedimento cl?ssico de descida de gradiente estoc?stico (SGD) para atualizar os pesos da rede de forma iterativa com base nos dados de treinamento.

![](https://cdn-images-1.medium.com/max/1600/1*X9gB3l_Wh5owNPCUsaYQVQ.png)

Mais informa??es: [artigo original](https://arxiv.org/abs/1412.6980), [post explicativo](https://towardsdatascience.com/adam-latest-trends-in-deep-learning-optimization-6be9a291375c), [outros otimizadores](http://ruder.io/optimizing-gradient-descent/)



> **Importante**: quando estiverem fazendo experimentos com NN, testem com SGD e Adam e com diferentes **LEARNING RATES**.


### 3. Treinamento

In [ ]:
model.fit(x=X_train, y=y_train, validation_data=(X_val,y_val), batch_size=16, epochs=10)

> Percebemos que s? com 10 ?pocas a rede ainda n?o tinha convergido, o loss ainda estava caindo, ent?o poder?amos treinar por mais ?pocas!

Temos dois par?metros importantes no treinamento:
- N?mero de ?pocas: Quantas vezes a rede vai passar por todos as inst?ncias
- Tamanho do batch: Qual o tamanho do bloco que ela vai usar, ou seja, quantas inst?ncias por vez passar?o pela rede


### 4. Avalia??o

In [ ]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
recall_score(y_test, y_pred)

In [ ]:
precision_score(y_test, y_pred)

Perceba que apesar da nossa acur?cia ser alta, a qualidade do modelo ? ruim com baixa sensibilidade. Vamos retreinar o mesmo modelo, desta vez passando peso para as classes.

In [ ]:
model = build_model()
model.compile(loss='binary_crossentropy',
             optimizer=adam,
             metrics=['accuracy'])



In [ ]:
model.fit(x=X_train, y=y_train, validation_data=(X_val,y_val), batch_size=16, epochs=10, class_weight={0:0.2,1:0.8})

In [ ]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5)

cm = confusion_matrix(y_test, y_pred)
print(cm)

## Exerc?cio:
Crie uma nova arquitetura de rede para o mesmo dataset. Mude tamb?m o learning rate e a quantidade de ?pocas e compare os resultados


In [ ]:
def build_model2():
    model = Sequential()

    model.add(Dense(units=8, input_dim=12, activation='relu'))
    model.add(Dense(8, activation='relu'))


    model.add(Dense(1, activation='sigmoid'))

    return model

In [ ]:
model2 = build_model2()
model2.summary()

In [ ]:
## TODO defina o learning rate
adam = Adam(lr=0.01)
model2.compile(loss='binary_crossentropy',
             optimizer=adam,
             metrics=['accuracy'])

In [ ]:
## TODO defina o total de ?pocas
model2.fit(x=X_train, y=y_train, validation_data=(X_val,y_val), batch_size=16, epochs=12, class_weight={0:0.2,1:0.8} )

In [ ]:
y_pred = model2.predict(X_test)
y_pred = (y_pred > 0.5)
cm = confusion_matrix(y_test, y_pred)
print(cm)

## Por que o crescimento de Deep Learning?

<img src="https://kevinzakka.github.io/assets/app_dl/perf_vs_data.png" alt="drawing" width="600"/>

Algoritmos tradicionais tendem a estabilizar a performance apartir de uma certa quantidade de dados, enquanto redes neurais tendem a ficar cada vez melhores quanto mais dados s?o utilizados para o aprendizado.

Portanto, o principal motivo que faz com que as NN cres?am nos ?ltimos anos ? o grande aumento na quantidade de **dados** dispon?veis.  Al?m disso, o poder **computacional** tamb?m ? muito maior nos dias atuais, principalmente com a utiliza??o de GPU's. O que tamb?m permitiu o desenvolvimento de **algoritmos** mais complexos e potentes.


Neural Networks, mais especificamente Deep Learning, tem grande aplica??es em datas n?o-estruturados, como: Imagens, A?dios e Textos.